## Importing IPP

In [24]:
import ipp

In [25]:
# model_Eval = ipp.pd.DataFrame()
ipp.warnings.filterwarnings("ignore")
ipp.initial_Check()

In [ ]:
df = ipp.pd.read_csv('/Users/nikhilprao/Documents/Data/Boston.csv', index_col=0)
df.reset_index(drop=True)

## ask user for file 
### for now its commented 

In [ ]:
# import tkinter as tk
# from tkinter import filedialog

# # Initialize tkinter window
# root = tk.Tk()
# root.withdraw()  # Hide the root window

# # Ask the user to upload any file
# file_path = filedialog.askopenfilename(title="Select a file")

# # Send the uploaded file to the function
# if file_path:
#     df = ipp.load_data(file_path)
#     print("Dataset loaded successfully!")
# else:
#     print("No file selected.")


## Pre processing unit will come here

In [ ]:
df.isnull().sum()
# applying the method
nan_in_df = df.isnull().sum().any()
 
# Print the dataframe
print(type(nan_in_df))

df.info()

df.describe()

### Ask User for predictive column

In [ ]:
# Display column names to the user
print("Available predictor variables:")
for idx, col in enumerate(df.columns):
    print(f"{idx + 1}. {col}")

# Ask the user to choose a predictor variable
selected_index = int(input("Enter the index of the predictor variable you want to choose: ")) - 1

# Validate user input
if 0 <= selected_index < len(df.columns):
    pattern = df.columns[selected_index]
    print(f"Selected predictor variable: {pattern}")
else:
    print("Invalid index selected. Please choose a valid index.")


predictor_variable = df.filter(regex=f'^{pattern}').columns[0]

## rename the pred column if required

In [ ]:
# # Rename the column if it contains the predictor_variable with a suffix (e.g., medv_power -> medv)
# def rename(df, predictor_variable):
#     return df.rename(columns={col: predictor_variable if predictor_variable in col else col for col in df.columns})

# df_skew = rename(df_skew, predictor_variable)

# # Print the modified DataFrame
# print(df_skew)

## Model starts here

In [ ]:
ipp.interModel(df, predictor_variable, "raw")

## Pipeline for Outliers

In [ ]:
df_outlier_cleaned = ipp.main_outliers(df, predictor_variable)

In [ ]:
df_outlier_cleaned

In [ ]:
ipp.interModel(df_outlier_cleaned, predictor_variable, "outliers")
# ipp.Comp_plot_boxplots(df, df_outlier_cleaned, predictor_variable)

## Viz

In [ ]:
# fig, axs = ipp.ipp.plt.subplots(ncols=7, nrows=2, figsize=(20, 10))
# index = 0
# axs = axs.flatten()
# for col, value in df_outlier_cleaned.items() :
#     ipp.ipp.sns.distplot(value, ax=axs[index])
#     index += 1
# ipp.ipp.plt. tight_layout (pad=0.5, w_pad=0.7, h_pad=5.0)

In [ ]:
# #box plot to all features to show outliers
# fig, axs = ipp.ipp.plt.subplots(ncols=7, nrows=2, figsize=(20, 10))
# index = 0
# axs = axs.flatten()
# for col, value in df_outlier_cleaned.items():
#     ipp.ipp.sns.boxplot(y=col, data=df_outlier_cleaned, ax=axs[index])
#     index += 1
# ipp.ipp.plt.tight_layout (pad=0.5, w_pad=0.7, h_pad=5.0)

In [ ]:
# ipp.plot_numerical_columns(df_outlier_cleaned)

In [ ]:
# ipp.sns.pairplot(df_outlier_cleaned)

## Co relation matrix

In [ ]:
corr_mat=df_outlier_cleaned.corr()
print(type(corr_mat))
# ipp.corrplot(corr_mat)

In [ ]:
unique_counts = df.nunique()
print(unique_counts)
df_outlier_cleaned.info()

## Adding the threshold from corr matrix for model

In [ ]:
df_filtered, high_loss = ipp.remove_high_correlation_features(df_outlier_cleaned,predictor_variable)
print(high_loss)
# Output: {'High_loss': {'feature3': 0.91, 'feature4': 0.95}, 'Threshold': 0.9}
# print(df_filtered.head(2))
ipp.update_high_correlation_features(high_loss["High_loss"])

In [ ]:
df_filtered.columns

In [ ]:
low_threshold_value = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55]
results = {}

for i in low_threshold_value:
    df_filtered, low_loss = ipp.remove_low_correlation_features(df_outlier_cleaned, i, predictor_variable)
    num_features = len(low_loss["Low_loss"])
    
    results[i] = {
        "df_filtered": df_filtered,  # Stores the dataframe
        "num_features": num_features  # Stores the number of low-correlation features removed
    }

    model_name = "LTH_"
    ipp.interModel(df_filtered, predictor_variable, model_name+str(i))
    ipp.update_low_correlation_features(i, low_loss["Low_loss"])
    # print(df_filtered.head(2))
    # print(" ----------------- ")

# Return the dictionary containing all results
# results


In [ ]:
df_04 = results[0.4]["df_filtered"]
df_04.head(2)

## Skew Handling

In [ ]:
# Define skewness thresholds
high_skew_threshold = 1
moderate_skew_threshold = 0.5

# Calculate skewness for all columns
skew_values = df_04.skew()

# Categorize columns based on skewness values
highly_skewed = skew_values[abs(skew_values) > high_skew_threshold].index.tolist()
moderately_skewed = skew_values[(abs(skew_values) >= moderate_skew_threshold) & (abs(skew_values) <= high_skew_threshold)].index.tolist()
low_skew = skew_values[abs(skew_values) < moderate_skew_threshold].index.tolist()

# Print categorized columns
print("Highly Skewed:", highly_skewed)
print("Moderately Skewed:", moderately_skewed)
print("Low Skew:", low_skew)

In [ ]:
df_outlier_cleaned.skew()

## Normality check of each data frame

In [ ]:
normality_results = ipp.check_normality(df_04)
print(normality_results)

## Skew Handling

In [ ]:
# Main function to apply skew handling
def skew_handling(df, highly_skewed, moderately_skewed):
    df = ipp.handle_high_skew(df, highly_skewed)  # Handling high skew
    ipp.interModel(df, predictor_variable, "High_skew")
    df = ipp.handle_moderate_skew(df, moderately_skewed)  # Handling moderate skew
    ipp.interModel(df, predictor_variable, "Moderate_skew")
    
    print("\n Skew Handling Completed. Returning Transformed DataFrame.")
    return df

# Example Usage:
# df_selected_3 = skew_handling(df_selected_3, highly_skewed, moderately_skewed)


In [ ]:
df_skew = skew_handling(df_04, highly_skewed, moderately_skewed)

In [ ]:
try:
    with open(ipp.json_file_path, "r") as file:
        status_data = ipp.json.load(file)
except FileNotFoundError:
    # If the file doesn't exist, initialize an empty structure
    status_data = {}

# Now you can safely modify the status_data object
status_data["pre_processing"]["Skew"]["Low"]["handling"] = False
status_data["pre_processing"]["Skew"]["Low"]["features"] = low_skew

# Write the updated status back to the file
with open(ipp.json_file_path, "w") as file:
    ipp.json.dump(status_data, file, indent=4)

## Lasso n Ridge with CV

In [ ]:
print(df_skew.head(2))
df_skew.skew()

# Hyper parameter tuning

In [ ]:
from sklearn.linear_model import LassoCV, Ridge, RidgeCV, ElasticNet, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split as ipp

# Function definitions for models (as in your provided code)
def Scaled_Linear_Model(X_train_scaled, y_train, X_test_scaled, y_test):
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "Scaled_Linear_Model"

def Lasso_model(X_train_scaled, y_train, X_test_scaled, y_test):
    lasso = Lasso()
    lasso.fit(X_train_scaled, y_train)
    y_train_pred = lasso.predict(X_train_scaled)
    y_test_pred = lasso.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "Lasso_model"

def LassoCV_model(X_train_scaled, y_train, X_test_scaled, y_test):
    lassocv = LassoCV(cv=5)
    lassocv.fit(X_train_scaled, y_train)
    y_train_pred = lassocv.predict(X_train_scaled)
    y_test_pred = lassocv.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "LassoCV_model"

def Ridge_model(X_train_scaled, y_train, X_test_scaled, y_test):
    ridge = Ridge()
    ridge.fit(X_train_scaled, y_train)
    y_train_pred = ridge.predict(X_train_scaled)
    y_test_pred = ridge.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "Ridge_model"

def RidgeCV_model(X_train_scaled, y_train, X_test_scaled, y_test):
    ridgecv = RidgeCV(cv=5)
    ridgecv.fit(X_train_scaled, y_train)
    y_train_pred = ridgecv.predict(X_train_scaled)
    y_test_pred = ridgecv.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "RidgeCV_model"

def ElasticNet_model(X_train_scaled, y_train, X_test_scaled, y_test):
    elastic = ElasticNet()
    elastic.fit(X_train_scaled, y_train)
    y_train_pred = elastic.predict(X_train_scaled)
    y_test_pred = elastic.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "ElasticNet_model"

def ElasticNetCV_model(X_train_scaled, y_train, X_test_scaled, y_test):
    elasticcv = ElasticNetCV(cv=5)
    elasticcv.fit(X_train_scaled, y_train)
    y_train_pred = elasticcv.predict(X_train_scaled)
    y_test_pred = elasticcv.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "ElasticNetCV_model"

In [ ]:
df_skew

In [ ]:
df_04.head(2)

In [ ]:
# Data Preparation
X = df_04.drop(columns=[predictor_variable])
y = df_04[predictor_variable]

X_train, X_test, y_train, y_test = ipp(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

In [ ]:
# List of models to evaluate
models = [Scaled_Linear_Model, Lasso_model, LassoCV_model, Ridge_model, RidgeCV_model, ElasticNet_model, ElasticNetCV_model]

# Initialize an empty dictionary to store model results
model_results = {}

# Evaluate each model and store the results
for model in models:
    train_score, test_score, r_squared, model_name = model(X_train_scaled, y_train, X_test_scaled, y_test)
    # Add model results to the dictionary in the desired format
    model_results[model_name] = {
        "train": train_score,
        "test": test_score,
        "r2_score": r_squared,
        "model": model_name
    }


In [ ]:
import json
import os

# Define the JSON file path (adjust this to your actual file path)
# Define the JSON file path (adjust this to your actual file path)
model_dir = "model_dump"
json_file_path = os.path.join(model_dir, "status.json")
data_folder = "data"

# Try to load the existing JSON data
try:
    with open(json_file_path, "r") as file:
        status_data = json.load(file)
except FileNotFoundError:
    # If the file doesn't exist, initialize an empty structure
    status_data = {}

# Ensure that 'modeling' and 'hyperparameter_tuning' keys exist in status_data
if 'modeling' not in status_data:
    status_data['modeling'] = {}

if 'hyperparameter_tuning' not in status_data:
    status_data['hyperparameter_tuning'] = {}

# Function to update the JSON with model results
def update_status_json(model_results):
    for model_name, metrics in model_results.items():
        # Check if the model is one of the special cases for hyperparameter tuning
        if 'CV' in model_name:  # Models with "CV" should go to hyperparameter_tuning
            status_data['hyperparameter_tuning'][model_name] = {
                "train": metrics['train'],
                "test": metrics['test'],
                "r2_score": metrics['r2_score'],
                "model": metrics['model'],
                "grid_search": False,
                "random_search": False
            }
        else:
            # Otherwise, the model is added to the "modeling" section
            status_data['modeling'][model_name] = {
                "train": metrics['train'],
                "test": metrics['test'],
                "r2_score": metrics['r2_score'],
                "model": metrics['model']
            }

# Update the status JSON with the model results
update_status_json(model_results)

# Write the updated data back to the file
with open(json_file_path, "w") as file:
    json.dump(status_data, file, indent=4)

print("Status JSON updated successfully.")


# KNeighborsRegressor
## Basic KNN

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score

def run_knn(df,target, standardize=False, tune=False):
    X = df.drop(columns=[target])
    y = df[target]
    results = {}

    # 1. Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 2. Standardize if needed
    if standardize:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    # 3. Model setup
    if tune:
        param_grid = {
            'n_neighbors': list(range(1, 31)),
            'weights': ['uniform', 'distance'],
            'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
            'p': [1, 2]
        }
        model = GridSearchCV(KNeighborsRegressor(), param_grid, cv=5, n_jobs=-1, scoring='r2')
    else:
        model = KNeighborsRegressor()

    # 4. Fit the model
    model.fit(X_train, y_train)

    # 5. Best model if tuned
    if tune:
        best_model = model.best_estimator_
    else:
        best_model = model

    # 6. Scores
    train_score = best_model.score(X_train, y_train)
    test_score = best_model.score(X_test, y_test)
    y_pred = best_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)

    # 7. Build results dictionary
    results = {
        "model_name": "KNeighborsRegressor_Tuned" if tune else "KNeighborsRegressor",
        "standardized": standardize,
        "train_score": train_score,
        "test_score": test_score,
        "r2_score": r2
    }
    if tune:
        results["best_params"] = model.best_params_

    return results


In [ ]:
import json
import os

# Define the JSON file path
model_dir = "model_dump"
json_file_path = os.path.join(model_dir, "status.json")

# Try to load the existing JSON data
try:
    with open(json_file_path, "r") as file:
        status_data = json.load(file)
except FileNotFoundError:
    # If the file doesn't exist, initialize an empty structure
    status_data = {}

def run_all_knn_variations(x, y):
    combinations = [
        {"standardize": False, "tune": False},
        {"standardize": True, "tune": False},
        {"standardize": False, "tune": True},
        {"standardize": True, "tune": True},
    ]

    all_results = []

    for combo in combinations:
        result = run_knn(x, y, standardize=combo["standardize"], tune=combo["tune"])
        all_results.append(result)

    return all_results


# Function to update the JSON with model results
def update_status_json(model_results):
    for metrics in model_results:
        model_name = metrics['model_name']
        standardized = metrics['standardized']

        # Build a cleaner name key
        full_model_name = f"{model_name} | Standardized: {standardized}"

        if 'Tuned' in model_name:  # Tuned models go to hyperparameter_tuning
            status_data['hyperparameter_tuning'][full_model_name] = {
                "train": metrics['train_score'],
                "test": metrics['test_score'],
                "r2_score": metrics['r2_score'],
                "best_params": metrics['best_params'],
                "grid_search": True,  # Since tuning is via grid search
                "random_search": False
            }
        else:  # Basic models go to modeling
            status_data['modeling'][full_model_name] = {
                "train": metrics['train_score'],
                "test": metrics['test_score'],
                "r2_score": metrics['r2_score']
            }

# Update the status JSON with the model results
update_status_json(run_all_knn_variations(df_skew, predictor_variable))

# Write the updated data back to the file
os.makedirs(model_dir, exist_ok=True)  # Ensure the folder exists
with open(json_file_path, "w") as file:
    json.dump(status_data, file, indent=4)

print("Status JSON updated successfully.")


## Dtree

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
from datetime import datetime

def run_decision_tree(df, target, tune=False, plot=True):
    X = df.drop(columns=[target])
    y = df[target]
    results = {}

    # 1. Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 2. Model setup
    if tune:
        param_grid = {
            'max_depth': [None, 1, 2, 3, 4, 5, 10, 20, 50],
            'criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson'],
            'splitter': ['best', 'random'],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None]  # Removed 'auto'
        }
        model = GridSearchCV(DecisionTreeRegressor(random_state=42), param_grid, cv=5, n_jobs=-1, scoring='r2')
    else:
        model = DecisionTreeRegressor(random_state=42)

    # 3. Fit the model
    model.fit(X_train, y_train)

    # 4. Best model if tuned
    best_model = model.best_estimator_ if tune else model

    # 5. Scores
    train_score = best_model.score(X_train, y_train)
    test_score = best_model.score(X_test, y_test)
    y_pred = best_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)

    # 6. Build results dictionary
    results = {
        "model_name": "DecisionTreeRegressor_Tuned" if tune else "DecisionTreeRegressor",
        "train_score": train_score,
        "test_score": test_score,
        "r2_score": r2,
        "model": best_model 
    }
    if tune:
        results["best_params"] = model.best_params_

    # 7. Plot the tree
    if plot:
        plt.figure(figsize=(40, 20))  # Bigger figure size
        plot_tree(
            best_model,
            feature_names=X.columns,
            filled=True,
            rounded=True
        )
        plt.title("Decision Tree - {}".format("Tuned" if tune else "Untuned"))
        # Generate a unique filename with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        plot_filename = f"tree_plot_{'tuned' if tune else 'untuned'}_{timestamp}.png"
        plt.savefig(plot_filename, dpi=300)  # High-res image
        plt.close()  # Close the plot after saving to avoid overlapping

    return results


In [ ]:
import json
import os
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# Paths
model_dir = "model_dump"
output_dir = os.path.join(model_dir, "Out_Put")
json_file_path = os.path.join(model_dir, "status.json")
os.makedirs(output_dir, exist_ok=True)

# Load or initialize JSON
try:
    with open(json_file_path, "r") as file:
        status_data = json.load(file)
except FileNotFoundError:
    status_data = {"modeling": {}, "hyperparameter_tuning": {}}

# Helper to save tree plot
def save_tree_plot(model, feature_names, filename):
    plt.figure(figsize=(40, 20))
    plot_tree(model, feature_names=feature_names, filled=True, rounded=True)
    plot_path = os.path.join(output_dir, filename)
    plt.savefig(plot_path, dpi=300)
    plt.close()
    return plot_path

# Run both models
untuned_result = run_decision_tree(df_skew, predictor_variable, tune=False, plot=False)
tuned_result = run_decision_tree(df_skew, predictor_variable, tune=True, plot=False)

# Save plots and add paths
X = df_skew.drop(columns=[predictor_variable])
untuned_plot_path = save_tree_plot(untuned_result['model'], X.columns, "dtree_untuned.png")
tuned_plot_path = save_tree_plot(tuned_result['model'], X.columns, "dtree_tuned.png")

# Update results
status_data["modeling"][untuned_result["model_name"]] = {
    "train": untuned_result["train_score"],
    "test": untuned_result["test_score"],
    "r2_score": untuned_result["r2_score"],
    "plot_path": untuned_plot_path
}
status_data["hyperparameter_tuning"][tuned_result["model_name"]] = {
    "train": tuned_result["train_score"],
    "test": tuned_result["test_score"],
    "r2_score": tuned_result["r2_score"],
    "best_params": tuned_result["best_params"],
    "grid_search": True,
    "random_search": False,
    "plot_path": tuned_plot_path
}

# Write to JSON
with open(json_file_path, "w") as file:
    json.dump(status_data, file, indent=4)

print("✅ Decision Tree results and plots saved successfully.")


In [ ]:
# # It auto-detects whether the tuning was done by GridSearchCV or RandomizedSearchCV or neither (plain model).
# #🚀 Full Upgraded Code:

# import json
# import os
# from datetime import datetime

# # Define the JSON file path
# model_dir = "model_dump"
# json_file_path = os.path.join(model_dir, "status.json")

# # Try to load the existing JSON data
# try:
#     with open(json_file_path, "r") as file:
#         status_data = json.load(file)
# except FileNotFoundError:
#     status_data = {}

# # Ensure required sections exist
# status_data.setdefault('modeling', {})
# status_data.setdefault('hyperparameter_tuning', {})

# # Sample model results you provided
# model_results = [
#     {'model_name': 'KNeighborsRegressor', 'standardized': False, 'train_score': 0.7429003113266298, 'test_score': 0.6156818824793304, 'r2_score': 0.6156818824793304},
#     {'model_name': 'KNeighborsRegressor', 'standardized': True, 'train_score': 0.854174924892116, 'test_score': 0.7773386665131938, 'r2_score': 0.7773386665131938},
#     {'model_name': 'KNeighborsRegressor_Tuned', 'standardized': False, 'train_score': 1.0, 'test_score': 0.6990520797886772, 'r2_score': 0.6990520797886772, 'best_params': {'algorithm': 'auto', 'n_neighbors': 4, 'p': 1, 'weights': 'distance'}, 'tuning_method': 'GridSearchCV'},
#     {'model_name': 'KNeighborsRegressor_Tuned', 'standardized': True, 'train_score': 1.0, 'test_score': 0.7657717802180388, 'r2_score': 0.7657717802180388, 'best_params': {'algorithm': 'auto', 'n_neighbors': 8, 'p': 1, 'weights': 'distance'}, 'tuning_method': 'RandomizedSearchCV'}
# ]

# # Function to update the JSON
# def update_status_json(model_results):
#     for metrics in model_results:
#         model_name = metrics['model_name']
#         standardized = metrics['standardized']

#         # Prepare a unique model key
#         full_model_name = f"{model_name} | Standardized: {standardized}"

#         # Timestamp when this entry is updated
#         timestamp_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

#         # Detect if it's tuned or not
#         if 'Tuned' in model_name or 'best_params' in metrics:
#             tuning_method = metrics.get('tuning_method', 'GridSearchCV')  # Default to GridSearchCV if not specified
#             status_data['hyperparameter_tuning'][full_model_name] = {
#                 "train": metrics['train_score'],
#                 "test": metrics['test_score'],
#                 "r2_score": metrics['r2_score'],
#                 "best_params": metrics['best_params'],
#                 "grid_search": tuning_method == 'GridSearchCV',
#                 "random_search": tuning_method == 'RandomizedSearchCV',
#                 "timestamp": timestamp_now
#             }
#         else:
#             # Plain modeling
#             status_data['modeling'][full_model_name] = {
#                 "train": metrics['train_score'],
#                 "test": metrics['test_score'],
#                 "r2_score": metrics['r2_score'],
#                 "timestamp": timestamp_now
#             }

# # Update JSON
# update_status_json(model_results)

# # Ensure folder exists
# os.makedirs(model_dir, exist_ok=True)

# # Write the updated JSON back
# with open(json_file_path, "w") as file:
#     json.dump(status_data, file, indent=4)

# print("Status JSON updated successfully.")


## Viz for model comparision

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
# Models and their corresponding R² scores
models = [
    "raw", "outliers", "LTH_0.3", "LTH_0.35", "LTH_0.4", "LTH_0.45", "LTH_0.5", "LTH_0.55",
    "High_skew", "Moderate_skew", "Scaled_Linear_Model", "Lasso_model", "Ridge_model", "ElasticNet_model",
    "LassoCV_model", "RidgeCV_model", "ElasticNetCV_model"
]

r2_scores = [
    0.6893967884614722, 0.7477564629456128, 0.7429717474588526, 0.7429717474588526,
    0.7429717474588526, 0.7060787046551034, 0.6726340566919277, 0.6670850446313604,
    0.7181137685011099, 0.7301638953107763, 0.726035785336048, 0.6669480378991592,
    0.725316016960783, 0.62765669023569, 0.7258860055606041, 0.725316016960783, 0.7244937556179412
]
# Create figure
plt.figure(figsize=(12, 7))

# Create improved bar plot
plt.figure(figsize=(12, 7))
colors = sns.color_palette("Blues_r", len(models))
# bars = plt.bar(models, r2_scores, color=colors, edgecolor="black")

# Bar plot
bars = plt.bar(models, r2_scores, color=colors, edgecolor="black", alpha=0.6, label="R² Score (Bar)")

# Line plot
plt.plot(models, r2_scores, marker="o", color="red", linestyle="-", linewidth=2, markersize=6, label="R² Score (Line)")

# Add value labels
for bar, score in zip(bars, r2_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{score:.3f}",
             ha="center", fontsize=10, fontweight="bold")

# Labels and Title
plt.ylabel("R² Score")
plt.title("Comparison of R² Scores for Different Models", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.ylim(0, 1)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Show plot
plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
json_file_path = os.path.join(model_dir, "status.json")

with open(json_file_path, "r") as f:
    data = json.load(f)

# Extract models and r2_scores
models = []
r2_scores = []

# From modeling
for model_name, model_info in data.get("modeling", {}).items():
    if isinstance(model_info, dict) and "r2_score" in model_info:
        models.append(model_name)
        r2_scores.append(model_info["r2_score"])

# From hyperparameter_tuning
for model_name, model_info in data.get("hyperparameter_tuning", {}).items():
    if isinstance(model_info, dict) and "r2_score" in model_info:
        models.append(model_name)
        r2_scores.append(model_info["r2_score"])

# Sort models by R² score (ascending)
model_scores = sorted(zip(models, r2_scores), key=lambda x: x[1])
models_sorted, r2_scores_sorted = zip(*model_scores)

# Color palette: last model = light green
base_colors = sns.color_palette("Blues", len(models_sorted)-1)
colors = list(base_colors) + [(0.5, 0.9, 0.5)]  # RGB for light green

# Plot
plt.figure(figsize=(14, 8))
bars = plt.bar(models_sorted, r2_scores_sorted, color=colors, edgecolor="black", alpha=0.7, label="R² Score (Bar)")
plt.plot(models_sorted, r2_scores_sorted, marker="o", color="black", linewidth=2, label="R² Score (Line)")

# Value labels
for bar, score in zip(bars, r2_scores_sorted):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015, f"{score:.3f}",
             ha="center", fontsize=10, fontweight="bold")

# Labels and aesthetics
plt.ylabel("R² Score", fontsize=12)
plt.title("Model and Tuned Model R² Score Comparison", fontsize=15, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Sample data: Replace with actual data
models = ["Raw", "Outliers", "LTH_0.3", "LTH_0.35", "LTH_0.4", "LTH_0.45", "LTH_0.5", "LTH_0.55", "High_skew", "Moderate_skew", "Scaled_Linear_Model", "Lasso_model", "Ridge_model", "ElasticNet_model", "LassoCV_model", "RidgeCV_model", "ElasticNetCV_model"]
r2_scores = [0.7398, 0.7478, 0.7430, 0.7429, 0.7429, 0.7061, 0.6726, 0.6671, 0.7181, 0.7302, 0.7260, 0.6670, 0.7253, 0.6277, 0.7259, 0.7253, 0.7245]  # Replace with actual R² scores
train_scores = [0.7398, 0.7514, 0.7507, 0.7507, 0.7507, 0.7244, 0.6708, 0.6702, 0.7353, 0.7376, 0.7381, 0.6673, 0.7381, 0.6628, 0.7381, 0.7381, 0.7379]  # Replace with actual train scores

# Create line plot for test (R² scores)
plt.figure(figsize=(12, 7))
plt.plot(models, r2_scores, marker="o", color="blue", linestyle="-", linewidth=2, markersize=6, label="Test R² Score")

# Create line plot for train scores
plt.plot(models, train_scores, marker=" ", color="red", linestyle="dotted", linewidth=2, markersize=6, label="Train R² Score")

# Add value labels to the test dots only
for i, score in enumerate(r2_scores):
    plt.text(models[i], score + 0.01, f"{score:.3f}", ha="center", fontsize=10, fontweight="bold", color="blue")

# Add labels and title
plt.ylabel("R² Score")
plt.title("Comparison of Train and Test R² Scores for Different Models", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.ylim(0, 1)
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Show the plot
plt.legend()
plt.show()


# Clean UP

In [ ]:
clean = bool(int(input("Enter 1 for Yes, 0 for No: ")))  # Converts 1 to True and 0 to False
if clean:
    ipp.initial_Check()
else: print("  BYE  ")

## Assumptions and some plots 

In [ ]:
import ipp

In [ ]:
X = df_skew.drop(columns=[predictor_variable])
y = df_skew[predictor_variable]

# Splitting the data into train and test sets
X_train, X_test, y_train, y_test = ipp.train_test_split(X, y, test_size=0.2, random_state=42)
model = ipp.LinearRegression()

In [ ]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

In [ ]:
Tar = y_train.name

In [ ]:
# ## standardize the dataset Train independent data
# from sklearn.preprocessing import StandardScaler

# scaler=StandardScaler()
# X_train=scaler.fit_transform(X_train)
# X_test=scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt

# Scale and rewrap to preserve DataFrame structure
scaler = StandardScaler()
feature_names = X_train.columns  # Save before transform

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_test = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

# Create scatter plots for all features vs target
num_features = X_train.shape[1]
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 12))
axes = axes.flatten()

for i in range(num_features):
    axes[i].scatter(X_train.iloc[:, i], y_train, alpha=0.6)
    axes[i].set_title(f'{X_train.columns[i]} vs Target')
    axes[i].set_xlabel(X_train.columns[i])
    axes[i].set_ylabel('Target')

for j in range(num_features, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
model.fit(X_train, y_train)
print("The slope or coefficient of weight is ",model.coef_)
print("Intercept:",model.intercept_)

In [ ]:
y_pred = model.predict(X_train)

plt.scatter(y_train, y_pred, alpha=0.6)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')  # Ideal line
plt.show()


In [ ]:
y_pred_test=model.predict(X_test)
plt.scatter(y_test, y_pred_test, alpha=0.6)

In [ ]:
y_pred = model.predict(X_test)

plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted (Test Data)")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # Perfect prediction line
plt.show()


In [ ]:
plt.scatter(y_test,y_pred_test)

In [ ]:
## Residuals
residuals=y_test-y_pred_test
residuals

In [ ]:
## plot this residuals
import seaborn as sns
sns.distplot(residuals,kde=True)

In [ ]:
plt.scatter(y_pred_test,residuals)


In [ ]:
mse=ipp.mean_squared_error(y_test,y_pred_test)
mae=ipp.mean_absolute_error(y_test,y_pred_test)
rmse=ipp.np.sqrt(mse)
print(mse)
print(mae)
print(rmse)

In [ ]:
score=ipp.r2_score(y_test,y_pred_test)
score

In [ ]:
# Data Preparation
X = df.drop(columns=[predictor_variable])
y = df[predictor_variable]

X_train, X_test, y_train, y_test = ipp.train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
plt.subplots(figsize=(15, 5))
plt.subplot(1, 2, 1)
ipp.sns.boxplot(data=X_train)
plt.title('X_train Before Scaling')
plt.subplot(1, 2, 2)
ipp.sns.boxplot(data=X_train_scaled)
plt.title('X_train After Scaling')

## NN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


X = df.drop('medv', axis=1).values
y = df['medv'].values.reshape(-1, 1)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

# Model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(12, 10)
        self.fc2 = nn.Linear(10, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(500):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Prediction
model.eval()
predicted = model(X_test).detach().numpy()
print("Actual:", y_test.numpy().flatten())
print("Predicted:", predicted.flatten())


In [ ]:
from sklearn.metrics import r2_score

# Calculate R² score
r2 = r2_score(y_test, predicted)
print(f"R² Score: {r2:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

# Load your dataset (assuming 'df' is your DataFrame with the Boston housing data)
X = df.drop('medv', axis=1).values
y = df['medv'].values.reshape(-1, 1)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

# Model with more layers
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(12, 50)  # Increase number of neurons in first hidden layer
        self.fc2 = nn.Linear(50, 25)  # Second hidden layer
        self.fc3 = nn.Linear(25, 1)   # Output layer

    def forward(self, x):
        x = torch.relu(self.fc1(x))  # ReLU activation for first hidden layer
        x = torch.relu(self.fc2(x))  # ReLU activation for second hidden layer
        return self.fc3(x)           # No activation for output layer

model = Net()

# L2 Regularization (Weight decay in SGD)
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=0.01)  # L2 regularization

# Loss function
criterion = nn.MSELoss()

# Training loop
for epoch in range(1000):  # Increase epochs for better training
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Prediction
model.eval()
predicted = model(X_test).detach().numpy()

# R² score
r2 = r2_score(y_test, predicted)
print(f"R² Score: {r2:.4f}")

# Display actual vs predicted
print("Actual:", y_test.numpy().flatten())
print("Predicted:", predicted.flatten())


In [ ]:
0.8669

# PDF

In [ ]:
# pip install fpdf2


In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# from fpdf import FPDF
# import os

# # ========================================
# # 1. Load Dataset (For testing)
# # ========================================



# # ========================================
# # 2. Generate Example Plots
# # ========================================
# # Create plots/ folder if it doesn't exist
# os.makedirs("plots", exist_ok=True)

# # Plot 1: Histogram of dataset
# df.hist(figsize=(10,8))
# plt.tight_layout()
# plt.savefig("plots/histogram.png")
# plt.close()

# # Plot 2: Correlation heatmap
# import seaborn as sns
# corr = df.corr(numeric_only=True)
# plt.figure(figsize=(8,6))
# sns.heatmap(corr, annot=True, cmap='coolwarm')
# plt.title("Correlation Heatmap")
# plt.tight_layout()
# plt.savefig("plots/heatmap.png")
# plt.close()

# # ========================================
# # 3. Create PDF with Dataset and Plots
# # ========================================
# class PDF(FPDF):
#     def header(self):
#         self.set_font("Arial", "B", 16)
#         self.cell(0, 10, "Testing ML Report", ln=True, align="C")
#         self.ln(10)

#     def chapter_title(self, title):
#         self.set_font("Arial", "B", 14)
#         self.cell(0, 10, title, ln=True)
#         self.ln(5)

#     def chapter_body(self, text):
#         self.set_font("Arial", "", 12)
#         self.multi_cell(0, 10, text)
#         self.ln()

#     def insert_image(self, img_path, w=150):
#         self.image(img_path, w=w)
#         self.ln(10)

# # Create PDF
# pdf = PDF()
# pdf.set_auto_page_break(auto=True, margin=15)
# pdf.add_page()

# # Add Introduction Section
# pdf.chapter_title("1. Introduction")
# intro_text = "This is a testing report generated with dataset description and visualizations."
# pdf.chapter_body(intro_text)

# # Add Dataset Information Section
# pdf.chapter_title("2. Dataset Information")
# describe_text = df.describe().to_string()
# pdf.chapter_body(describe_text)

# # Add Visualizations Section
# pdf.chapter_title("3. Visualizations")
# for plot_file in os.listdir("plots"):
#     if plot_file.endswith(".png"):
#         pdf.insert_image(f"plots/{plot_file}")

# # Save the PDF locally
# pdf.output("test_report.pdf")

# print("Test PDF generated successfully as 'test_report.pdf'.")


# work

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [2]:
df = pd.read_csv('/Users/nikhilprao/Documents/Data/Boston.csv', index_col=0)
df.reset_index(drop=True)

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,6.48,22.0


In [3]:
df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
1,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
2,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
3,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,4.03,34.7
4,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
5,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2


In [4]:
## Getting All Different Types OF Features
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('Num of Numerical Features :', len(num_features))
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of Categorical Features :', len(cat_features))
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Discrete Features :',len(discrete_features))
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('Num of Continuous Features :',len(continuous_features))

Num of Numerical Features : 13
Num of Categorical Features : 0
Num of Discrete Features : 2
Num of Continuous Features : 11


In [5]:
## Indpendent and dependent features
from sklearn.model_selection import train_test_split

X = df.drop(['medv'], axis=1)
y = df['medv']

In [6]:
X.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat
1,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98
2,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14
3,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,4.03
4,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94
5,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33


In [7]:
for column in df.columns:
    print(f"Column: {column}")
    print(f"Unique Values: {df[column].unique()}")
    print(df[column].value_counts())
    print("-" * 50)

Column: crim
Unique Values: [6.32000e-03 2.73100e-02 2.72900e-02 3.23700e-02 6.90500e-02 2.98500e-02
 8.82900e-02 1.44550e-01 2.11240e-01 1.70040e-01 2.24890e-01 1.17470e-01
 9.37800e-02 6.29760e-01 6.37960e-01 6.27390e-01 1.05393e+00 7.84200e-01
 8.02710e-01 7.25800e-01 1.25179e+00 8.52040e-01 1.23247e+00 9.88430e-01
 7.50260e-01 8.40540e-01 6.71910e-01 9.55770e-01 7.72990e-01 1.00245e+00
 1.13081e+00 1.35472e+00 1.38799e+00 1.15172e+00 1.61282e+00 6.41700e-02
 9.74400e-02 8.01400e-02 1.75050e-01 2.76300e-02 3.35900e-02 1.27440e-01
 1.41500e-01 1.59360e-01 1.22690e-01 1.71420e-01 1.88360e-01 2.29270e-01
 2.53870e-01 2.19770e-01 8.87300e-02 4.33700e-02 5.36000e-02 4.98100e-02
 1.36000e-02 1.31100e-02 2.05500e-02 1.43200e-02 1.54450e-01 1.03280e-01
 1.49320e-01 1.71710e-01 1.10270e-01 1.26500e-01 1.95100e-02 3.58400e-02
 4.37900e-02 5.78900e-02 1.35540e-01 1.28160e-01 8.82600e-02 1.58760e-01
 9.16400e-02 1.95390e-01 7.89600e-02 9.51200e-02 1.01530e-01 8.70700e-02
 5.64600e-02 8.38700e-0

In [8]:
num_features = X.select_dtypes(exclude="object").columns
onehot_columns = X.select_dtypes(include="object").columns

In [9]:
onehot_columns

Index([], dtype='object')

In [10]:
# Create Column Transformer with 3 types of transformers
num_features = X.select_dtypes(exclude="object").columns
onehot_columns = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("StandardScaler", numeric_transformer, num_features)
        
    ],remainder='passthrough'
    
)

In [11]:
X=preprocessor.fit_transform(X)

In [12]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11
0,-0.419782,0.284830,-1.287909,-0.272599,-0.144217,0.413672,-0.120013,0.140214,-0.982843,-0.666608,-1.459000,-1.075562
1,-0.417339,-0.487722,-0.593381,-0.272599,-0.740262,0.194274,0.367166,0.557160,-0.867883,-0.987329,-0.303094,-0.492439
2,-0.417342,-0.487722,-0.593381,-0.272599,-0.740262,1.282714,-0.265812,0.557160,-0.867883,-0.987329,-0.303094,-1.208727
3,-0.416750,-0.487722,-1.306878,-0.272599,-0.835284,1.016303,-0.809889,1.077737,-0.752922,-1.106115,0.113032,-1.361517
4,-0.412482,-0.487722,-1.306878,-0.272599,-0.835284,1.228577,-0.511180,1.077737,-0.752922,-1.106115,0.113032,-1.026501
...,...,...,...,...,...,...,...,...,...,...,...,...
501,-0.413229,-0.487722,0.115738,-0.272599,0.158124,0.439316,0.018673,-0.625796,-0.982843,-0.803212,1.176466,-0.418147
502,-0.415249,-0.487722,0.115738,-0.272599,0.158124,-0.234548,0.288933,-0.716639,-0.982843,-0.803212,1.176466,-0.500850
503,-0.413447,-0.487722,0.115738,-0.272599,0.158124,0.984960,0.797449,-0.773684,-0.982843,-0.803212,1.176466,-0.983048
504,-0.407764,-0.487722,0.115738,-0.272599,0.158124,0.725672,0.736996,-0.668437,-0.982843,-0.803212,1.176466,-0.865302


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((404, 12), (102, 12))

In [14]:
X_train

array([[ 1.32780421, -0.48772236,  1.01599907, ...,  1.53092646,
         0.80657583,  1.7181012 ],
       [-0.34750602, -0.48772236, -0.43725801, ..., -0.6012761 ,
         1.17646583, -0.5863558 ],
       [-0.41648392,  1.01446252, -0.74074945, ..., -0.61909395,
        -0.71922039, -0.67606702],
       ...,
       [-0.41877066,  2.94584308, -1.3316823 , ..., -0.76163674,
        -0.67298414, -0.93398678],
       [ 0.87825441, -0.48772236,  1.01599907, ...,  1.53092646,
         0.80657583,  1.48821619],
       [-0.39389588, -0.48772236, -0.37597609, ..., -0.14395131,
         1.13022958, -0.28358043]])

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [17]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [18]:
## Beginning Model Training
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "Adaboost Regressor":AdaBoostRegressor(),
    "Graident BoostRegressor":GradientBoostingRegressor(),
    "Xgboost Regressor":XGBRegressor()
   
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 4.7544
- Mean Absolute Error: 3.4245
- R2 Score: 0.7398
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 4.7726
- Mean Absolute Error: 3.1114
- R2 Score: 0.6894


Lasso
Model performance for Training set
- Root Mean Squared Error: 5.3417
- Mean Absolute Error: 3.7388
- R2 Score: 0.6715
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.2278
- Mean Absolute Error: 3.4504
- R2 Score: 0.6273


Ridge
Model performance for Training set
- Root Mean Squared Error: 4.7545
- Mean Absolute Error: 3.4205
- R2 Score: 0.7398
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 4.7763
- Mean Absolute Error: 3.1075
- R2 Score: 0.6889


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 3.6993
- Mean Absolute Error: 2.4330
- R2 Score: 0.8425
-----------------------

In [19]:
#Initialize few parameter for Hyperparamter tuning

rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

xgboost_params = {"learning_rate": [0.1, 0.01],
                  "max_depth": [5, 8, 12, 20, 30],
                  "n_estimators": [100, 200, 300],
                  "colsample_bytree": [0.5, 0.8, 1, 0.3, 0.4]}

In [20]:
# Models list for Hyperparameter tuning
randomcv_models = [
                   ("RF", RandomForestRegressor(), rf_params),
                   ("XGboost",XGBRegressor(),xgboost_params)
                   
                   ]

In [21]:
##Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END max_depth=8, max_features=5, min_samples_split=15, n_estimators=500; total time=   0.3s
[CV] END max_depth=8, max_features=5, min_samples_split=15, n_estimators=500; total time=   0.3s
[CV] END max_depth=8, max_features=5, min_samples_split=15, n_estimators=500; total time=   0.3s
[CV] END max_depth=5, max_features=5, min_samples_split=8, n_estimators=100; total time=   0.1s
[CV] END max_depth=5, max_features=5, min_samples_split=8, n_estimators=100; total time=   0.1s
[CV] END max_depth=5, max_features=5, min_samples_split=8, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, max_features=5, min_samples_split=15, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, max_features=5, min_samples_split=15, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, max_features=5, min_samples_split=15, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, max_features=auto, min_samples_split=2, n

---------------- Best Params for RF -------------------
{'n_estimators': 1000, 'min_samples_split': 2, 'max_features': 5, 'max_depth': None}
---------------- Best Params for XGboost -------------------
{'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.3}

In [22]:
## Retraining the models with best parameters
models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=1000, min_samples_split=2, max_features=5, max_depth=None, 
                                                     n_jobs=-1),
     "Xgboost Regressor":XGBRegressor(n_estimators= 200,learning_rate=0.1,
                                     max_depth=5,colsample_bytree=0.3)
    
}
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 1.2662
- Mean Absolute Error: 0.8265
- R2 Score: 0.9815
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 2.9408
- Mean Absolute Error: 1.9269
- R2 Score: 0.8821


Xgboost Regressor
Model performance for Training set
- Root Mean Squared Error: 0.5349
- Mean Absolute Error: 0.4024
- R2 Score: 0.9967
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 3.1321
- Mean Absolute Error: 2.0699
- R2 Score: 0.8662




## Need to add this for easy time period 